In [50]:
import json
import os

os.makedirs("data/jsons", exist_ok = True)

json_data = {
    "company" : "Apple Inc",
    "Employees" : [
        {
            "Employee_id" : 1,
            "Employee_name" : "Jaywardhan Pagar",
            "Employee_role" : "Data Scientist",
            "Salary" : "$15,000/month",
            "Skills" : ["Git","Machine Learning","RAG"],
            "Projects" : [
                {"Name" : "RAG Based AI Teaching Assistant", "Status" : "Completed"},
                {"Name" : "RAG Based Document Managament System", "Status" : "In Progress"}
            ]

        },

        {
            "Employee_id" : 2,
            "Employee_name" : "Anurag Mhaske",
            "Employee_role" : "Software Developer",
            "Salary" : "$10,000/month",
            "Skills" : ["Swift","Git","Kivy"],
            "Projects" : [
                {"Name" : "App Store", "Status" : "Completed"},
                {"Name" : "Siri AI", "Status" : "In Progress"}
            ]
        
        }


    ],

    "Departments" : [
        {
            "Department" : "Product Design",
            "Head" : "Steve Wozniak",
            "Budget" : "$7500000",
            "team_size" : 25
        },

        {
            "Department" : "Software Testing",
            "Head" : "Mike Ermentraut",
            "Budget" : "$5500000",
            "team_size" : 10
        }
    ]
}

with open("data/jsons/employees_data.json","w") as f:
    json.dump(json_data,f, indent = 2)

In [51]:
json_data1 = [
    {"timestamp" : "2026-08-08", "event" : "User Login", "user_id" : 113},
    {"timestamp" : "2026-08-08", "event" : "Page View", "user_id" : 113, "page" : "/services"},
    {"timestamp" : "2026-08-08", "event" : "Purchase", "user_id" : 113, "amount" : "$499"}
]

with open("data/jsons/events.jsonl","w") as f:
    for item in json_data1:
        f.write(json.dumps(item) + '\n')

Json data parsing


In [52]:
from langchain_community.document_loaders import JSONLoader
import json

try:

    employee_loader = JSONLoader(
        file_path = 'data/jsons/employees_data.json',
        jq_schema = '.Employees[]', # jq query to extract each employee,
        text_content = False # To get full json content
    )

    employee_docs = employee_loader.load()

    print(f"Total employee document are: {len(employee_docs)}")

    for i , emp in enumerate(employee_docs):
        print(f"Employee: {i+1}")
        print(f"Info: \n{emp.page_content}")
        print(f"Metadata: {emp.metadata}")

except Exception as e:
    print(f"Error: {e}")

Total employee document are: 2
Employee: 1
Info: 
{"Employee_id": 1, "Employee_name": "Jaywardhan Pagar", "Employee_role": "Data Scientist", "Salary": "$15,000/month", "Skills": ["Git", "Machine Learning", "RAG"], "Projects": [{"Name": "RAG Based AI Teaching Assistant", "Status": "Completed"}, {"Name": "RAG Based Document Managament System", "Status": "In Progress"}]}
Metadata: {'source': '/home/jaywardhan/RAG_Udemy/Data-Ingestion&Parsing/data/jsons/employees_data.json', 'seq_num': 1}
Employee: 2
Info: 
{"Employee_id": 2, "Employee_name": "Anurag Mhaske", "Employee_role": "Software Developer", "Salary": "$10,000/month", "Skills": ["Swift", "Git", "Kivy"], "Projects": [{"Name": "App Store", "Status": "Completed"}, {"Name": "Siri AI", "Status": "In Progress"}]}
Metadata: {'source': '/home/jaywardhan/RAG_Udemy/Data-Ingestion&Parsing/data/jsons/employees_data.json', 'seq_num': 2}


In [53]:
from typing import List
from langchain_core.documents import Document
def customJSON(filepath: str) -> List[Document]:

    with open(filepath , 'r') as f:
        emp_doc = json.load(f)

    documents = []
    for emp in emp_doc.get('Employees', '[]'):

        content = f"""Employee Information: 
            "Employee ID": {emp["Employee_id"]}
            "Name": {emp["Employee_name"]}
            "Role": {emp["Employee_role"]}
            "Salary": {emp["Salary"]}
            "Skills": {emp["Skills"]}
           
            "Projects": """
        
        for proj in emp.get('Projects', []):
            content += f"\n - {proj["Name"]} : Status({proj["Status"]})"
                    
        

        doc = Document(
            page_content = content,
            metadata = {
                "Employee ID" : emp['Employee_id'],
                "Name" : emp["Employee_name"],
                "Role" : emp["Employee_role"],
                "Salary" : emp["Salary"],
                "Skills" : emp["Skills"],
                "Projects" : emp["Projects"],
                "Source" : filepath
                
            }
        )

        documents.append(doc)

    return documents

try:

    json_documents = customJSON("data/jsons/employees_data.json")

    print(f"Number of docs: {len(json_documents)}")
    for i , doc in enumerate(json_documents):
        print(f"Document: {i + 1}")
        print(f"Info: {doc.page_content}")
        print(f"Metadata: {doc.metadata}")

except Exception as e:
    print(f"Error: {e}")    

Number of docs: 2
Document: 1
Info: Employee Information: 
            "Employee ID": 1
            "Name": Jaywardhan Pagar
            "Role": Data Scientist
            "Salary": $15,000/month
            "Skills": ['Git', 'Machine Learning', 'RAG']

            "Projects": 
 - RAG Based AI Teaching Assistant : Status(Completed)
 - RAG Based Document Managament System : Status(In Progress)
Metadata: {'Employee ID': 1, 'Name': 'Jaywardhan Pagar', 'Role': 'Data Scientist', 'Salary': '$15,000/month', 'Skills': ['Git', 'Machine Learning', 'RAG'], 'Projects': [{'Name': 'RAG Based AI Teaching Assistant', 'Status': 'Completed'}, {'Name': 'RAG Based Document Managament System', 'Status': 'In Progress'}], 'Source': 'data/jsons/employees_data.json'}
Document: 2
Info: Employee Information: 
            "Employee ID": 2
            "Name": Anurag Mhaske
            "Role": Software Developer
            "Salary": $10,000/month
            "Skills": ['Swift', 'Git', 'Kivy']

            "Project

In [54]:
try:

    events_loader = JSONLoader(
    file_path = "data/jsons/events.jsonl",
    jq_schema = ".",
    text_content = False,
    json_lines = True
    )

    events_docs = events_loader.load()

    print(events_docs)

except Exception as e:
    print(f"Error: {e}")

[Document(metadata={'source': '/home/jaywardhan/RAG_Udemy/Data-Ingestion&Parsing/data/jsons/events.jsonl', 'seq_num': 1}, page_content='{"timestamp": "2026-08-08", "event": "User Login", "user_id": 113}'), Document(metadata={'source': '/home/jaywardhan/RAG_Udemy/Data-Ingestion&Parsing/data/jsons/events.jsonl', 'seq_num': 2}, page_content='{"timestamp": "2026-08-08", "event": "Page View", "user_id": 113, "page": "/services"}'), Document(metadata={'source': '/home/jaywardhan/RAG_Udemy/Data-Ingestion&Parsing/data/jsons/events.jsonl', 'seq_num': 3}, page_content='{"timestamp": "2026-08-08", "event": "Purchase", "user_id": 113, "amount": "$499"}')]


Custom processing .jsonl file:

In [87]:
def customJSONL(filepath: str) -> List[Document]:

    events_doc = []
    with open("data/jsons/events.jsonl", "r") as f:
       events_jsons = [json.loads(line) for line in f]

    for i , event in enumerate(events_jsons):

        content = f"""
        "Timestamp": {event['timestamp']},
        "Event": {event['event']},
        "User Id": {event['user_id']}
        "Page": {event.get('page','Not visited')}
        "Amount" : {event.get('amount','nil')}
        """

        doc = Document(
            page_content = content,
            metadata = {
                'source' : filepath,
                'user_id' : event['user_id'],
                'event' : event['event'],
                'timestamp' : event['timestamp'],
                "Page": event.get('page','Not visited'),
                "Amount" : event.get('amount','nil'),
                'event_no' : i + 1
            }
        )

        events_doc.append(doc)

    return events_doc


        

In [88]:
docs = customJSONL("data/jsons/events.jsonl")

for i , event in enumerate(docs):
    print(f"Doc: {i+1}")
    print(f"{event.page_content}")
    print(f"metadata: {event.metadata}")

Doc: 1

        "Timestamp": 2026-08-08,
        "Event": User Login,
        "User Id": 113
        "Page": Not visited
        "Amount" : nil
        
metadata: {'source': 'data/jsons/events.jsonl', 'user_id': 113, 'event': 'User Login', 'timestamp': '2026-08-08', 'Page': 'Not visited', 'Amount': 'nil', 'event_no': 1}
Doc: 2

        "Timestamp": 2026-08-08,
        "Event": Page View,
        "User Id": 113
        "Page": /services
        "Amount" : nil
        
metadata: {'source': 'data/jsons/events.jsonl', 'user_id': 113, 'event': 'Page View', 'timestamp': '2026-08-08', 'Page': '/services', 'Amount': 'nil', 'event_no': 2}
Doc: 3

        "Timestamp": 2026-08-08,
        "Event": Purchase,
        "User Id": 113
        "Page": Not visited
        "Amount" : $499
        
metadata: {'source': 'data/jsons/events.jsonl', 'user_id': 113, 'event': 'Purchase', 'timestamp': '2026-08-08', 'Page': 'Not visited', 'Amount': '$499', 'event_no': 3}


In [89]:
print(docs)

[Document(metadata={'source': 'data/jsons/events.jsonl', 'user_id': 113, 'event': 'User Login', 'timestamp': '2026-08-08', 'Page': 'Not visited', 'Amount': 'nil', 'event_no': 1}, page_content='\n        "Timestamp": 2026-08-08,\n        "Event": User Login,\n        "User Id": 113\n        "Page": Not visited\n        "Amount" : nil\n        '), Document(metadata={'source': 'data/jsons/events.jsonl', 'user_id': 113, 'event': 'Page View', 'timestamp': '2026-08-08', 'Page': '/services', 'Amount': 'nil', 'event_no': 2}, page_content='\n        "Timestamp": 2026-08-08,\n        "Event": Page View,\n        "User Id": 113\n        "Page": /services\n        "Amount" : nil\n        '), Document(metadata={'source': 'data/jsons/events.jsonl', 'user_id': 113, 'event': 'Purchase', 'timestamp': '2026-08-08', 'Page': 'Not visited', 'Amount': '$499', 'event_no': 3}, page_content='\n        "Timestamp": 2026-08-08,\n        "Event": Purchase,\n        "User Id": 113\n        "Page": Not visited\n   